# Упражнения к семинару 11. Проверка нормальности распределения

Переименуйте файл в формате `Группа-Фамилия-Имя-seminar-11-tasks.ipynb`, например `2MP9-Ivanov-Ivan-seminar-11-tasks.ipynb`: этот файл сдаётся в Moodle. Формулировки упражнений здесь те же, что в листке `seminar-11-tasks.md`; дополнительно в ноутбуке заготовки функций, тесты и ячейки «Проверка».

Данные: таблицы Московской области за 2022–2025 годы, ДТП `datasets/legal/dtp-moscow-region.csv` и участники `datasets/legal/dtp-moscow-region-participants.csv`; путь в ячейках задан от папки `exercises/`, поэтому ноутбук должен лежать в ней, а папка `datasets` рядом, как в репозитории.

- Часть А: выполните ячейку с подключением библиотек, соберите таблицу водителей в упражнении А1, затем решайте упражнения по порядку и вводите числа и решения в тест Moodle «Семинар 11».
- Части Б и В: напишите тело функции вместо `pass`, запустите ячейку с тестами; если всё верно, она напечатает `Все тесты прошли успешно!`. Затем выполните ячейку «Проверка»: она вызывает функцию на таблицах области и печатает числа, которые вводятся в тест вместе с кодом функции.

Правила: функция не ссылается на переменные, определённые вне неё, кроме `pd`, `np`, `plt`, `sns`, `stats`, `lilliefors` и функций части Б; ячейки с тестами не меняются; в коде решений библиотеки, кроме pandas, NumPy, matplotlib, seaborn, scipy и statsmodels, не используются.

## Часть А. Разминка

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from statsmodels.stats.diagnostic import lilliefors

sns.set_theme(style='whitegrid')       # оформление графиков по умолчанию
pd.set_option('display.width', 200)   # широкие таблицы печатаются без переноса

### А1. Форма распределения стажа

Загрузите таблицу ДТП `datasets/legal/dtp-moscow-region.csv` в `df` с `index_col='id'` и `parse_dates=['datetime']` и таблицу участников `datasets/legal/dtp-moscow-region-participants.csv` в `people`. Соберите таблицу водителей с их ДТП, как в разделе 1 ноутбука занятия: стаж без кодов 95, 96 и 99 в новом столбце `people['experience_clean']`; `drivers = people[people['role'] == 'Водитель'].merge(df, left_on='dtp_id', right_index=True)` — должно получиться 25 364 водителя; возраст машины `drivers['vehicle_age']` — год ДТП минус `vehicle_year`, значения больше 100 — пропуск через `where`. Возьмите стаж без пропусков `experience = drivers['experience_clean'].dropna()` и посчитайте асимметрию, эксцесс и доли значений в пределах одного и двух стандартных отклонений от среднего. Каковы асимметрия и эксцесс, до сотых? Каковы доли в пределах одного и двух отклонений, до тысячных? Подсказка: раздел 1 ноутбука занятия; асимметрия — `skew()`, эксцесс — `kurt()`, у нормального распределения оба нули; удаление от среднего — `(experience - experience.mean()).abs()`, доля — среднее маски `<= experience.std()`; у нормального распределения доли 0,683 и 0,954. Стаж с кодами, столбец `experience`, даёт другие числа: коды 95–99 — длинный искусственный хвост. Доли, посчитанные по столбцу `drivers['experience_clean']` с пропусками, занижены: сравнение с пропуском даёт `False`, а пропуск попадает в знаменатель.

In [ ]:
# А1

### А2. Гистограмма с кривой и Q–Q график

На двух панелях `fig, axes = plt.subplots(1, 2, figsize=(12, 4))` нарисуйте слева гистограмму стажа в плотностях с кривой нормального распределения с теми же средним и отклонением: `sns.histplot(x=experience, stat='density', discrete=True, ax=axes[0])`, сетка `grid = np.linspace(experience.min(), experience.max(), 200)` и `axes[0].plot(grid, stats.norm.pdf(grid, loc=experience.mean(), scale=experience.std()), color='red')`; справа Q–Q график `stats.probplot(experience, plot=axes[1])`; подпишите панели и оси. Где гистограмма выше кривой, где ниже и куда уходит правый конец Q–Q графика? Посчитайте долю водителей со стажем выше среднего плюс два отклонения и долю водителей со стажем ниже среднего минус два отклонения: у нормального распределения обе доли по 0,023. Каковы эти доли, до тысячных? Подсказка: разделы 1 и 2 ноутбука занятия; верхняя граница — `experience.mean() + 2 * experience.std()`, доля — среднее маски `experience > граница`. Справа хвост толще нормального, слева хвоста нет вовсе: стаж не бывает отрицательным, — это и есть асимметрия из А1, показанная графиком. Доли по столбцу `drivers['experience_clean']` с пропусками занижены; без `dropna()` Q–Q график рисуется без прямой.

In [ ]:
# А2

### А3. Тест Шапиро–Уилка

Правило курса: тест Шапиро–Уилка идёт по всем значениям, если их не больше 5 000; если больше — тест Д’Агостино по всем значениям или Шапиро–Уилк на случайной выборке из 500 значений с `random_state=1`. Значений стажа больше 5 000, поэтому выборка: `stats.shapiro(experience.sample(500, random_state=1))`; затем тот же тест на выборке из 30 значений с тем же `random_state=1`. Запишите решение каждого теста словами при уровне значимости 0,05. Каково W на выборке 500, до тысячных, и каково решение? Каково p на выборке 30, до тысячных, и каково решение? Подсказка: раздел 4 ноутбука занятия; поля результата — `statistic` и `pvalue`; решение — `'отвергаем' if p < 0.05 else 'не отвергаем'`. Тест по всей таблице выдаёт предупреждение scipy о неточности p при n > 5000; выборка из столбца `drivers['experience_clean']` без `dropna()` содержит пропуски, и тест возвращает `nan`. «Не отвергаем» на 30 значениях не значит, что стаж нормален: малая выборка не видит отклонения.

In [ ]:
# А3

### А4. Тесты Д’Агостино и Лиллиефорса

Тест Д’Агостино по всему стажу без пропусков: `stats.normaltest(experience)`; тест Лиллиефорса: `lilliefors(experience)` — кортеж из статистики D и p-значения. Для сравнения выполните и `stats.kstest(experience, 'norm')`. Какова статистика теста Д’Агостино, до целого, и каково решение? Какова статистика D теста Лиллиефорса, до тысячных? Подсказка: раздел 4 ноутбука занятия. p Лиллиефорса 0,001 — нижняя граница таблицы statsmodels, а не точное значение. `kstest` без параметров сравнивает стаж со стандартным нормальным распределением — средним 0 и отклонением 1 — и отвергает что угодно: его статистика — не ответ.

In [ ]:
# А4

### А5. Преобразования

Возраст машины без пропусков: `age = drivers['vehicle_age'].dropna()`. Посчитайте асимметрию `age`, `np.log1p(age)` и `np.sqrt(age)` и нарисуйте три гистограммы рядом, как в разделе 6 ноутбука занятия. Какова асимметрия до преобразования, после логарифма и после корня, до сотых? Какое преобразование ближе к симметрии? Подсказка: раздел 6 ноутбука занятия; ближе к симметрии то, у которого асимметрия наименьшая по модулю. `np.log(age)` вместо `np.log1p(age)` даёт минус бесконечность у машин с возрастом 0, и асимметрия становится `nan`; логарифм поджимает хвост слишком сильно, и асимметрия меняет знак.

In [ ]:
# А5

### А6. Центральная предельная теорема

Тысяча средних по 100 водителей: `rng = np.random.default_rng(1)`; `means = rng.choice(experience.to_numpy(), size=(1000, 100)).mean(axis=1)`. Посчитайте разброс средних `means.std()`, σ/√n — `experience.std() / np.sqrt(100)` — и асимметрию средних `pd.Series(means).skew()`; нарисуйте гистограмму средних; выполните тест Шапиро–Уилка по всем 1 000 средним, `stats.shapiro(means)`. Каковы разброс средних и σ/√n, до тысячных? Какова асимметрия средних, до сотых, и каково решение теста? Подсказка: раздел 5 ноутбука занятия; генератор создаётся в той же ячейке, что и вытягивание: если выполнить `rng.choice` второй раз без нового `default_rng(1)`, выборки будут другими. Средних 1 000 — не больше 5 000, и по правилу курса тест Шапиро–Уилка идёт по всем значениям, без выборки. У самого стажа асимметрия из А1, у средних — почти ноль: нормальным становится распределение среднего, а не значений.

In [ ]:
# А6

### А7. Таблица нормальности

Таблица нормальности трёх столбцов, как в разделе 7 ноутбука занятия: стаж `experience`, возраст машины `age` и час ДТП из таблицы ДТП `hour = df['datetime'].dt.hour` — одно значение на ДТП. Для каждого — n, асимметрия, p теста Д’Агостино по всем значениям, решение при 0,05 и что дальше; таблицу удобно собрать из словаря `{'experience_clean': experience, 'vehicle_age': age, 'hour': hour}`. У какого столбца асимметрия наименьшая по модулю и какова она, до сотых? У скольких из трёх столбцов тест не отвергает нормальность? Подсказка: раздел 7 ноутбука занятия; наименьшая по модулю — `abs()` и `idxmin()`, в ответ идёт асимметрия со знаком. Час, посчитанный по таблице водителей, повторяет ДТП с несколькими машинами и даёт другую асимметрию. p, выведенное нулём, — число меньше самого малого, которое хранит компьютер; в отчёте пишут «p < 0,001». «Что дальше» — по таблице раздела 6 ноутбука занятия; для часа — с оговоркой, как в разделе 7: шкала циклическая, за 23 часами идёт 0, и час сравнивают долями, а не средними.

In [ ]:
# А7

## Часть Б. Основные задачи

### Б1. Сводка формы распределения

С такой сводки начинается раздел «Нормальность» этапа 3: семь чисел описывают форму распределения ещё до тестов. У нормального распределения асимметрия и эксцесс — нули, в пределах одного стандартного отклонения от среднего лежат 68,3 % значений, в пределах двух — 95,4 %. Пропуски в расчёт не входят, и доли считаются среди известных значений; значение ровно на расстоянии одного отклонения — в пределах.

Реализуйте функцию `normal_summary(values)`, которая считает числа формы распределения: размер, среднее, отклонение, асимметрию, эксцесс и доли по правилу 68–95. Аргументы: `values` — столбец чисел, Series; пропуски допускаются. Возвращаемое значение: объект Series из семи чисел с именами «n», «среднее», «отклонение», «асимметрия», «эксцесс», «в пределах одного отклонения», «в пределах двух отклонений» в этом порядке: число значений без пропусков; среднее; стандартное отклонение — std() pandas; асимметрия — skew(); эксцесс — kurt(); доли значений, удалённых от среднего не больше чем на одно и на два стандартных отклонения, среди значений без пропусков. Series не меняется.

Подсказка: раздел 1 ноутбука занятия: `values = values.dropna()` в начале функции; удаление от среднего — `(values - mean).abs()`, доля — среднее маски `distance <= std`; `pd.Series({'n': ..., 'среднее': ..., ...})` собирает числа с именами. `np.std` делит на n, а не на n − 1, и даёт другое отклонение; `stats.skew` и `stats.kurtosis` из scipy считают без поправки на размер выборки, и на маленьких данных числа другие.

Проверка в ноутбуке: стаж водителей-женщин области через normal_summary; печатается сводка, в тест вводятся эксцесс и доля в пределах двух отклонений.

In [ ]:
def normal_summary(values):
    """
    Считает числа формы распределения: размер, среднее, отклонение, асимметрию,
    эксцесс и доли по правилу 68–95.

    С такой сводки начинается раздел «Нормальность» этапа 3: семь чисел
    описывают форму распределения ещё до тестов. У нормального распределения
    асимметрия и эксцесс — нули, в пределах одного стандартного отклонения от
    среднего лежат 68,3 % значений, в пределах двух — 95,4 %. Пропуски в расчёт
    не входят, и доли считаются среди известных значений; значение ровно на
    расстоянии одного отклонения — в пределах.

    Аргументы:
        values: Столбец чисел, Series; пропуски допускаются.

    Возвращаемое значение:
        Объект Series из семи чисел с именами «n», «среднее», «отклонение»,
        «асимметрия», «эксцесс», «в пределах одного отклонения», «в пределах
        двух отклонений» в этом порядке: число значений без пропусков; среднее;
        стандартное отклонение — std() pandas; асимметрия — skew(); эксцесс —
        kurt(); доли значений, удалённых от среднего не больше чем на одно и на
        два стандартных отклонения, среди значений без пропусков. Series не
        меняется.
    """

    pass

In [ ]:
def normal_summary_tests():
    values = pd.Series([3, 3, 5, 7, 7, None], index=list('abcdef'))
    before = values.copy()
    result = normal_summary(values)
    names = ['n', 'среднее', 'отклонение', 'асимметрия', 'эксцесс', 'в пределах одного отклонения', 'в пределах двух отклонений']
    assert isinstance(result, pd.Series), 'функция должна возвращать Series'
    assert result.index.tolist() == names, 'имена строк как в условии, в том же порядке'
    assert result['n'] == 5, 'n — число значений без пропусков'
    assert result['среднее'] == 5 and result['отклонение'] == 2, 'отклонение — std() pandas, с делением на n − 1'
    assert result['в пределах одного отклонения'] == 1, 'значения 3 и 7 ровно на расстоянии одного отклонения: <=, а не <; пропуск не в знаменателе'
    skewed = normal_summary(pd.Series([1, 2, 2, 3, 3, 3, 4, 6, 9, None]))
    assert round(skewed['асимметрия'], 3) == 1.48 and round(skewed['эксцесс'], 3) == 2.165, 'асимметрия и эксцесс — skew() и kurt() pandas'
    assert round(skewed['в пределах одного отклонения'], 3) == 0.778, 'доля в пределах одного отклонения среди значений без пропусков'
    assert round(skewed['в пределах двух отклонений'], 3) == 0.889, 'доля в пределах двух отклонений среди значений без пропусков'
    assert values.equals(before), 'Series изменился'

    print('Все тесты прошли успешно!')

In [ ]:
normal_summary_tests()

In [ ]:
# Проверка: стаж водителей-женщин области через normal_summary; печатается сводка, в тест вводятся эксцесс и доля в пределах двух отклонений

df_check = pd.read_csv('../datasets/legal/dtp-moscow-region.csv', index_col='id', parse_dates=['datetime'])
people_check = pd.read_csv('../datasets/legal/dtp-moscow-region-participants.csv')
people_check['experience_clean'] = people_check['experience'].replace([95, 96, 99], np.nan)
drivers_check = people_check[people_check['role'] == 'Водитель'].merge(df_check, left_on='dtp_id', right_index=True)
drivers_check['vehicle_age'] = drivers_check['datetime'].dt.year - drivers_check['vehicle_year']
drivers_check['vehicle_age'] = drivers_check['vehicle_age'].where(drivers_check['vehicle_age'] <= 100)

women = drivers_check[drivers_check['gender'] == 'Женский']
print(normal_summary(women['experience_clean']).round(3))

### Б2. Гистограмма с кривой нормального распределения

Первый график раздела «Нормальность»: колокол с теми же средним и стандартным отклонением поверх гистограммы показывает, где данные расходятся с нормальным распределением. Чтобы высоты были сравнимы, гистограмма строится в плотностях: площадь всех столбцов — единица, как площадь под кривой. Кривая — 200 точек от наименьшего значения до наибольшего.

Бины зависят от значений. Целые значения — годы, штуки — рисуются столбцом на каждое целое, `discrete=True`, как стаж в разделе 1 ноутбука занятия: бины по умолчанию ложатся между целыми, часть столбцов выходит пустой, и гистограмма превращается в гребёнку. Целые — по значениям, а не по типу столбца: возраст машины с пропусками хранится дробными числами. Если хоть одно значение дробное — бины по умолчанию. Столбцов при `discrete=True` столько, сколько целых от наименьшего значения до наибольшего: для годов и счётчиков это десятки, а для сумм в рублях такой гистограммы не строят.

Реализуйте функцию `hist_with_normal(values, title, xlabel, ax=None)`, которая рисует гистограмму плотности и кривую нормального распределения с теми же средним и отклонением. Аргументы: `values` — столбец чисел, Series; пропуски допускаются; `title` — заголовок, строка; `xlabel` — подпись оси X, строка; `ax` — оси для рисования; если не переданы, функция создаёт свои: fig, ax = plt.subplots(figsize=(8, 4)). Возвращаемое значение: объект Axes с гистограммой values в плотностях — stat='density'; если все значения без пропусков целые, discrete=True — столбец шириной 1 на каждое целое от наименьшего значения до наибольшего, иначе бины по умолчанию — и одной линией: кривой нормального распределения со средним и стандартным отклонением values, посчитанной в 200 точках от наименьшего значения до наибольшего; заголовок title, подпись оси X xlabel, подпись оси Y «Плотность». Пропуски не рисуются. Series не меняется.

Подсказка: раздел 1 ноутбука занятия: после `dropna()` — целые ли значения, `whole = (values % 1 == 0).all()`; `sns.histplot(x=values, stat='density', discrete=whole, ax=ax)`; сетка `grid = np.linspace(values.min(), values.max(), 200)`; высоты кривой `stats.norm.pdf(grid, loc=values.mean(), scale=values.std())`, линия `ax.plot(grid, ...)`; проверка `if ax is None:` в начале функции. Проверка типа столбца целых с пропусками не найдёт: у них тип `float64`. `kde=True` добавит вторую линию.

Проверка в ноутбуке: возраст машины водителей области через hist_with_normal; печатаются число столбцов и высота кривой в вершине до тысячных, оба вводятся в тест.

In [ ]:
def hist_with_normal(values, title, xlabel, ax=None):
    """
    Рисует гистограмму плотности и кривую нормального распределения с теми же
    средним и отклонением.

    Первый график раздела «Нормальность»: колокол с теми же средним и
    стандартным отклонением поверх гистограммы показывает, где данные расходятся
    с нормальным распределением. Чтобы высоты были сравнимы, гистограмма
    строится в плотностях: площадь всех столбцов — единица, как площадь под
    кривой. Кривая — 200 точек от наименьшего значения до наибольшего.

    Бины зависят от значений. Целые значения — годы, штуки — рисуются столбцом
    на каждое целое, `discrete=True`, как стаж в разделе 1 ноутбука занятия:
    бины по умолчанию ложатся между целыми, часть столбцов выходит пустой, и
    гистограмма превращается в гребёнку. Целые — по значениям, а не по типу
    столбца: возраст машины с пропусками хранится дробными числами. Если хоть
    одно значение дробное — бины по умолчанию. Столбцов при `discrete=True`
    столько, сколько целых от наименьшего значения до наибольшего: для годов и
    счётчиков это десятки, а для сумм в рублях такой гистограммы не строят.

    Аргументы:
        values: Столбец чисел, Series; пропуски допускаются.
        title: Заголовок, строка.
        xlabel: Подпись оси X, строка.
        ax: Оси для рисования; если не переданы, функция создаёт свои: fig, ax =
            plt.subplots(figsize=(8, 4)).

    Возвращаемое значение:
        Объект Axes с гистограммой values в плотностях — stat='density'; если
        все значения без пропусков целые, discrete=True — столбец шириной 1 на
        каждое целое от наименьшего значения до наибольшего, иначе бины по
        умолчанию — и одной линией: кривой нормального распределения со средним
        и стандартным отклонением values, посчитанной в 200 точках от
        наименьшего значения до наибольшего; заголовок title, подпись оси X
        xlabel, подпись оси Y «Плотность». Пропуски не рисуются. Series не
        меняется.
    """

    pass

In [ ]:
def hist_with_normal_tests():
    values = pd.Series([1, 2, 2, 3, 3, 3, 4, 4, 5, 9, None])
    before = values.copy()
    ax = hist_with_normal(values, 'Заголовок', 'Значение')
    assert isinstance(ax, plt.Axes), 'функция должна возвращать Axes'
    area = sum(p.get_width() * p.get_height() for p in ax.patches)
    assert abs(area - 1) < 1e-6, "гистограмма в плотностях: площадь столбцов — единица, stat='density'"
    assert len(ax.patches) == 9 and all(p.get_width() == 1 for p in ax.patches), 'целые значения: столбец шириной 1 на каждое целое от 1 до 9, discrete=True'
    assert len(ax.lines) == 1, 'на осях одна линия — кривая нормального распределения'
    x, y = ax.lines[0].get_xdata(), ax.lines[0].get_ydata()
    assert len(x) == 200 and x[0] == 1 and x[-1] == 9, 'кривая в 200 точках от наименьшего значения до наибольшего'
    assert np.allclose(y, stats.norm.pdf(x, loc=3.6, scale=2.2211), atol=1e-4), 'кривая со средним 3.6 и отклонением std() pandas 2.2211'
    assert ax.get_title() == 'Заголовок' and ax.get_xlabel() == 'Значение' and ax.get_ylabel() == 'Плотность', 'заголовок title, подпись оси X xlabel, подпись оси Y «Плотность»'
    assert values.equals(before), 'Series изменился'
    fractions = pd.Series([0.5, 1.25, 2.0, 2.0, 2.75, 3.5, 3.5, 3.5, 4.25, 9.0, None])
    ax = hist_with_normal(fractions, 'Заголовок', 'Значение')
    assert len(ax.patches) == 7, 'есть дробные значения: бины по умолчанию, семь столбцов, без discrete и bins'
    assert abs(sum(p.get_width() * p.get_height() for p in ax.patches) - 1) < 1e-6, 'и здесь плотность'
    fig, own = plt.subplots()
    assert hist_with_normal(values, 'Заголовок', 'Значение', ax=own) is own, 'переданные оси должны использоваться'
    plt.close('all')

    print('Все тесты прошли успешно!')

In [ ]:
hist_with_normal_tests()

In [ ]:
# Проверка: возраст машины водителей области через hist_with_normal; печатаются число столбцов и высота кривой в вершине до тысячных, оба вводятся в тест

df_check = pd.read_csv('../datasets/legal/dtp-moscow-region.csv', index_col='id', parse_dates=['datetime'])
people_check = pd.read_csv('../datasets/legal/dtp-moscow-region-participants.csv')
people_check['experience_clean'] = people_check['experience'].replace([95, 96, 99], np.nan)
drivers_check = people_check[people_check['role'] == 'Водитель'].merge(df_check, left_on='dtp_id', right_index=True)
drivers_check['vehicle_age'] = drivers_check['datetime'].dt.year - drivers_check['vehicle_year']
drivers_check['vehicle_age'] = drivers_check['vehicle_age'].where(drivers_check['vehicle_age'] <= 100)

ax = hist_with_normal(drivers_check['vehicle_age'], 'Возраст машины водителей', 'Возраст, лет')
print(len(ax.patches), round(max(ax.lines[0].get_ydata()), 3))

### Б3. Q–Q график

Q–Q график точнее гистограммы показывает хвосты: точки на прямой — распределение близко к нормальному, правый конец вверх — длинный правый хвост. `stats.probplot` рисует на переданных осях точки и прямую. По столбцу с пропусками он не падает, а рисует без прямой, поэтому пропуски убираются заранее. Подписи осей scipy ставит по-английски, функция меняет их на русские.

Реализуйте функцию `qq_plot(values, title, ax=None)`, которая рисует Q–Q график значений против нормального распределения. Аргументы: `values` — столбец чисел, Series; пропуски допускаются; `title` — заголовок, строка; `ax` — оси для рисования; если не переданы, функция создаёт свои: fig, ax = plt.subplots(figsize=(5, 5)). Возвращаемое значение: объект Axes с Q–Q графиком значений values без пропусков, нарисованным stats.probplot: две линии — точки и прямая; заголовок title, подпись оси X «Квантили нормального распределения», подпись оси Y «Значения». Series не меняется.

Подсказка: раздел 2 ноутбука занятия: `stats.probplot(values.dropna(), plot=ax)`, затем `ax.set_title`, `ax.set_xlabel`, `ax.set_ylabel`; возвращаются оси `ax`, а не результат `probplot`.

Проверка в ноутбуке: час ДТП области через qq_plot; печатаются число точек и наклон прямой до сотых, оба вводятся в тест.

In [ ]:
def qq_plot(values, title, ax=None):
    """
    Рисует Q–Q график значений против нормального распределения.

    Q–Q график точнее гистограммы показывает хвосты: точки на прямой —
    распределение близко к нормальному, правый конец вверх — длинный правый
    хвост. `stats.probplot` рисует на переданных осях точки и прямую. По столбцу
    с пропусками он не падает, а рисует без прямой, поэтому пропуски убираются
    заранее. Подписи осей scipy ставит по-английски, функция меняет их на
    русские.

    Аргументы:
        values: Столбец чисел, Series; пропуски допускаются.
        title: Заголовок, строка.
        ax: Оси для рисования; если не переданы, функция создаёт свои: fig, ax =
            plt.subplots(figsize=(5, 5)).

    Возвращаемое значение:
        Объект Axes с Q–Q графиком значений values без пропусков, нарисованным
        stats.probplot: две линии — точки и прямая; заголовок title, подпись оси
        X «Квантили нормального распределения», подпись оси Y «Значения». Series
        не меняется.
    """

    pass

In [ ]:
def qq_plot_tests():
    values = pd.Series([4, 1, None, 3, 10, 2])
    before = values.copy()
    ax = qq_plot(values, 'Заголовок')
    assert isinstance(ax, plt.Axes), 'функция должна возвращать Axes'
    assert len(ax.lines) == 2, 'две линии: точки и прямая'
    points, line = ax.lines
    assert points.get_ydata().tolist() == [1, 2, 3, 4, 10], 'точки — значения по возрастанию, без пропусков'
    assert np.allclose(points.get_xdata(), [-1.129, -0.4857, 0, 0.4857, 1.129], atol=1e-3), 'по оси X — квантили стандартного нормального распределения для пяти значений'
    assert np.isfinite(line.get_ydata()).all(), 'прямая посчитана по значениям без пропусков'
    assert ax.get_title() == 'Заголовок', 'заголовок title'
    assert ax.get_xlabel() == 'Квантили нормального распределения' and ax.get_ylabel() == 'Значения', 'подписи осей по-русски'
    assert values.equals(before), 'Series изменился'
    fig, own = plt.subplots()
    assert qq_plot(values, 'Заголовок', ax=own) is own, 'переданные оси должны использоваться'
    plt.close('all')

    print('Все тесты прошли успешно!')

In [ ]:
qq_plot_tests()

In [ ]:
# Проверка: час ДТП области через qq_plot; печатаются число точек и наклон прямой до сотых, оба вводятся в тест

df_check = pd.read_csv('../datasets/legal/dtp-moscow-region.csv', index_col='id', parse_dates=['datetime'])

ax = qq_plot(df_check['datetime'].dt.hour, 'Час ДТП')
x, y = ax.lines[1].get_xdata(), ax.lines[1].get_ydata()
print(len(ax.lines[0].get_xdata()), round((y[-1] - y[0]) / (x[-1] - x[0]), 2))

### Б4. Шапиро–Уилк по правилу курса

Правило курса для теста Шапиро–Уилка: если значений без пропусков не больше 5 000, тест идёт по всем; если больше — на случайной выборке из n значений, по умолчанию 500, с `random_state=seed`, чтобы числа воспроизводились. Порог 5 000 — граница scipy: на большем числе значений он предупреждает, что p неточно; от n порог не зависит, и при 600 значениях выборка не нужна. Другой путь для больших таблиц — тест Д’Агостино по всем значениям, функция Б5. Пропуски убираются до подсчёта и до выборки: иначе они попадут в выборку, и тест вернёт `nan`.

Реализуйте функцию `shapiro_sample(values, n=500, seed=1)`, которая выполняет тест Шапиро–Уилка по правилу курса: по всем значениям или на случайной выборке. Аргументы: `values` — столбец чисел, Series; пропуски допускаются; `n` — размер выборки, целое число; по умолчанию 500; нужен, только если значений больше 5000; `seed` — зерно выборки для random_state, целое число; по умолчанию 1. Возвращаемое значение: кортеж из двух чисел (W, p): статистика и p-значение stats.shapiro по всем значениям values без пропусков, если их не больше 5000; если больше — на выборке values.dropna().sample(n, random_state=seed). Series не меняется.

Подсказка: раздел 4 ноутбука занятия: `values = values.dropna()`; `if len(values) > 5000:` — выборка `values.sample(n, random_state=seed)`; `result = stats.shapiro(values)`, `return result.statistic, result.pvalue`. Сравнивается число значений с 5000, а не с n; ровно 5000 значений — ещё тест по всем.

Проверка в ноутбуке: возраст машины водителей области и городского округа Домодедово через shapiro_sample; печатаются число значений, W и p, в тест вводятся оба W и решения при 0,05.

In [ ]:
def shapiro_sample(values, n=500, seed=1):
    """
    Выполняет тест Шапиро–Уилка по правилу курса: по всем значениям или на
    случайной выборке.

    Правило курса для теста Шапиро–Уилка: если значений без пропусков не больше
    5 000, тест идёт по всем; если больше — на случайной выборке из n значений,
    по умолчанию 500, с `random_state=seed`, чтобы числа воспроизводились. Порог
    5 000 — граница scipy: на большем числе значений он предупреждает, что p
    неточно; от n порог не зависит, и при 600 значениях выборка не нужна. Другой
    путь для больших таблиц — тест Д’Агостино по всем значениям, функция Б5.
    Пропуски убираются до подсчёта и до выборки: иначе они попадут в выборку, и
    тест вернёт `nan`.

    Аргументы:
        values: Столбец чисел, Series; пропуски допускаются.
        n: Размер выборки, целое число; по умолчанию 500; нужен, только если
           значений больше 5000.
        seed: Зерно выборки для random_state, целое число; по умолчанию 1.

    Возвращаемое значение:
        Кортеж из двух чисел (W, p): статистика и p-значение stats.shapiro по
        всем значениям values без пропусков, если их не больше 5000; если больше
        — на выборке values.dropna().sample(n, random_state=seed). Series не
        меняется.
    """

    pass

In [ ]:
def shapiro_sample_tests():
    values = pd.Series([2.0, 3, 5, 1, None, 8, 13, 4, 6, 9, 7, 30, None, 11, 12])
    before = values.copy()
    result = shapiro_sample(values, n=8, seed=2)
    assert isinstance(result, tuple) and len(result) == 2, 'функция должна возвращать кортеж (W, p)'
    w, p = result
    assert round(w, 4) == 0.7957 and round(p, 4) == 0.0061, 'значений не больше 5000: (W, p) теста по всем значениям без пропусков; n и seed не нужны'
    assert values.equals(before), 'Series изменился'
    big = pd.Series(np.sqrt(np.arange(6000.0)))
    big[::100] = None                                    # 5940 значений без пропусков
    before = big.copy()
    assert round(shapiro_sample(big)[0], 4) == 0.9407, 'больше 5000 значений: выборка values.dropna().sample(500, random_state=1)'
    assert round(shapiro_sample(big, n=300, seed=1)[0], 4) == 0.9422, 'размер выборки — n'
    assert round(shapiro_sample(big, n=300, seed=2)[0], 4) == 0.9428, 'зерно выборки — seed'
    assert big.equals(before), 'Series изменился'
    edge = big.iloc[:5051]                               # 5000 значений и 51 пропуск
    assert round(shapiro_sample(edge)[0], 4) == 0.9469, 'ровно 5000 значений без пропусков — тест по всем: пропуски не в счёт'

    print('Все тесты прошли успешно!')

In [ ]:
shapiro_sample_tests()

In [ ]:
# Проверка: возраст машины водителей области и городского округа Домодедово через shapiro_sample; печатаются число значений, W и p, в тест вводятся оба W и решения при 0,05

df_check = pd.read_csv('../datasets/legal/dtp-moscow-region.csv', index_col='id', parse_dates=['datetime'])
people_check = pd.read_csv('../datasets/legal/dtp-moscow-region-participants.csv')
people_check['experience_clean'] = people_check['experience'].replace([95, 96, 99], np.nan)
drivers_check = people_check[people_check['role'] == 'Водитель'].merge(df_check, left_on='dtp_id', right_index=True)
drivers_check['vehicle_age'] = drivers_check['datetime'].dt.year - drivers_check['vehicle_year']
drivers_check['vehicle_age'] = drivers_check['vehicle_age'].where(drivers_check['vehicle_age'] <= 100)

w, p = shapiro_sample(drivers_check['vehicle_age'])
print(drivers_check['vehicle_age'].count(), round(w, 3), p)
town = drivers_check[drivers_check['district'] == 'Домодедово']
w, p = shapiro_sample(town['vehicle_age'])
print(town['vehicle_age'].count(), round(w, 3), p)

### Б5. Таблица нормальности

Таблица раздела «Нормальность» этапа 3: строка на столбец, решение словами. Тест Д’Агостино идёт по всем значениям столбца без пропусков при любом их числе: выборка ему не нужна и на больших таблицах, поэтому в таблице один тест для всех строк. Решение — «отвергаем», если p меньше alpha, иначе «не отвергаем»; слова «нормально» в таблице нет: «не отвергаем» не значит «доказали».

Реализуйте функцию `normality_table(df, columns, alpha=0.05)`, которая собирает таблицу нормальности по столбцам: n, асимметрия, p теста Д’Агостино и решение. Аргументы: `df` — таблица; `columns` — список имён числовых столбцов; `alpha` — уровень значимости, число; по умолчанию 0.05. Возвращаемое значение: объект DataFrame со строками по columns в том же порядке и столбцами «n» — число значений без пропусков, «асимметрия» — skew() pandas, «p» — p-значение stats.normaltest по значениям без пропусков, «решение» — строка «отвергаем», если p < alpha, иначе «не отвергаем». Таблица не меняется.

Подсказка: раздел 7 ноутбука занятия; таблицу удобно начать с `pd.DataFrame(index=columns)` и добавлять по столбцу — список чисел по столбцам из `columns`, как в `outlier_table` семинара 10; p — `stats.normaltest(df[column].dropna()).pvalue`; решение — `np.where(table['p'] < alpha, 'отвергаем', 'не отвергаем')`. Без `dropna()` p — `nan`, а сравнение с `nan` даёт «не отвергаем».

Проверка в ноутбуке: водители городского округа Красноармейск через normality_table по стажу и возрасту машины; печатается таблица, в тест вводятся оба p до тысячных и оба решения.

In [ ]:
def normality_table(df, columns, alpha=0.05):
    """
    Собирает таблицу нормальности по столбцам: n, асимметрия, p теста Д’Агостино
    и решение.

    Таблица раздела «Нормальность» этапа 3: строка на столбец, решение словами.
    Тест Д’Агостино идёт по всем значениям столбца без пропусков при любом их
    числе: выборка ему не нужна и на больших таблицах, поэтому в таблице один
    тест для всех строк. Решение — «отвергаем», если p меньше alpha, иначе «не
    отвергаем»; слова «нормально» в таблице нет: «не отвергаем» не значит
    «доказали».

    Аргументы:
        df: Таблица.
        columns: Список имён числовых столбцов.
        alpha: Уровень значимости, число; по умолчанию 0.05.

    Возвращаемое значение:
        Объект DataFrame со строками по columns в том же порядке и столбцами «n»
        — число значений без пропусков, «асимметрия» — skew() pandas, «p» —
        p-значение stats.normaltest по значениям без пропусков, «решение» —
        строка «отвергаем», если p < alpha, иначе «не отвергаем». Таблица не
        меняется.
    """

    pass

In [ ]:
def normality_table_tests():
    a = [31, 37, 40, 42, 44, 45, 47, 48, 49, 50, 50, 51, 52, 53, 55, 56, 58, 60, 63, 69, None, None, None]
    b = [1] * 12 + [2, 2, 3, 3, 4, 5, 8, 13, 21, 34, None]
    df = pd.DataFrame({'a': a, 'b': b, 'name': list('uvwxyzuvwxyzuvwxyzuvwxy')})
    before = df.copy()
    table = normality_table(df, ['a', 'b'])
    assert isinstance(table, pd.DataFrame), 'функция должна возвращать DataFrame'
    assert table.index.tolist() == ['a', 'b'], 'строки — столбцы из списка columns'
    assert table.columns.tolist() == ['n', 'асимметрия', 'p', 'решение'], 'столбцы как в условии'
    assert table['n'].tolist() == [20, 22], 'n — число значений без пропусков'
    assert table['асимметрия'].round(3).tolist() == [0.0, 2.801], 'асимметрия — skew() pandas'
    assert round(table.loc['a', 'p'], 3) == 0.498 and table.loc['b', 'p'] < 0.001, 'p теста Д’Агостино, stats.normaltest, по значениям без пропусков'
    assert table['решение'].tolist() == ['не отвергаем', 'отвергаем'], 'решение: «отвергаем», если p < alpha'
    assert normality_table(df, ['a', 'b'], alpha=0.9)['решение'].tolist() == ['отвергаем', 'отвергаем'], 'уровень значимости alpha'
    assert normality_table(df, ['b', 'a']).index.tolist() == ['b', 'a'], 'строки в порядке columns'
    assert df.equals(before), 'таблица изменилась'

    print('Все тесты прошли успешно!')

In [ ]:
normality_table_tests()

In [ ]:
# Проверка: водители городского округа Красноармейск через normality_table по стажу и возрасту машины; печатается таблица, в тест вводятся оба p до тысячных и оба решения

df_check = pd.read_csv('../datasets/legal/dtp-moscow-region.csv', index_col='id', parse_dates=['datetime'])
people_check = pd.read_csv('../datasets/legal/dtp-moscow-region-participants.csv')
people_check['experience_clean'] = people_check['experience'].replace([95, 96, 99], np.nan)
drivers_check = people_check[people_check['role'] == 'Водитель'].merge(df_check, left_on='dtp_id', right_index=True)
drivers_check['vehicle_age'] = drivers_check['datetime'].dt.year - drivers_check['vehicle_year']
drivers_check['vehicle_age'] = drivers_check['vehicle_age'].where(drivers_check['vehicle_age'] <= 100)

town = drivers_check[drivers_check['district'] == 'Красноармейск']
print(normality_table(town, ['experience_clean', 'vehicle_age']))

### Б6. Тест Лиллиефорса

Тест Колмогорова–Смирнова из scipy, `stats.kstest(x, 'norm')`, сравнивает данные со стандартным нормальным распределением — средним 0 и отклонением 1 — и отвергает любой столбец в годах или рублях. Тест Лиллиефорса из statsmodels сравнивает с нормальным распределением, у которого среднее и отклонение оценены по данным. Его p берётся из таблицы, от 0,001 до 0,99: за этими границами точнее тест не покажет, и 0,001 значит «0,001 или меньше».

Реализуйте функцию `lilliefors_test(values)`, которая выполняет тест Лиллиефорса на нормальность. Аргументы: `values` — столбец чисел, Series; пропуски допускаются. Возвращаемое значение: кортеж из двух чисел (D, p): статистика и p-значение lilliefors с параметрами по умолчанию по значениям values без пропусков. Series не меняется.

Подсказка: раздел 4 ноутбука занятия: `lilliefors(values.dropna())` уже возвращает кортеж из двух чисел; полей `statistic` и `pvalue`, как у результатов scipy, у него нет.

Проверка в ноутбуке: возраст машины водителей области и городского округа Красноармейск через lilliefors_test; печатаются D и p обоих до тысячных, в тест вводятся D по области, p по городскому округу и решение по городскому округу.

In [ ]:
def lilliefors_test(values):
    """
    Выполняет тест Лиллиефорса на нормальность.

    Тест Колмогорова–Смирнова из scipy, `stats.kstest(x, 'norm')`, сравнивает
    данные со стандартным нормальным распределением — средним 0 и отклонением 1
    — и отвергает любой столбец в годах или рублях. Тест Лиллиефорса из
    statsmodels сравнивает с нормальным распределением, у которого среднее и
    отклонение оценены по данным. Его p берётся из таблицы, от 0,001 до 0,99: за
    этими границами точнее тест не покажет, и 0,001 значит «0,001 или меньше».

    Аргументы:
        values: Столбец чисел, Series; пропуски допускаются.

    Возвращаемое значение:
        Кортеж из двух чисел (D, p): статистика и p-значение lilliefors с
        параметрами по умолчанию по значениям values без пропусков. Series не
        меняется.
    """

    pass

In [ ]:
def lilliefors_test_tests():
    values = pd.Series([2.0, 3, 5, 1, None, 8, 13, 4, 6, 9, 7, 30, None, 11, 12])
    before = values.copy()
    result = lilliefors_test(values)
    assert isinstance(result, tuple) and len(result) == 2, 'функция должна возвращать кортеж (D, p)'
    d, p = result
    assert round(d, 4) == 0.1985 and round(p, 4) == 0.1833, 'тест Лиллиефорса по значениям без пропусков'
    d2, p2 = lilliefors_test(pd.Series([4.0, 5, 5, 6, 6, 6, 7, 7, 8, None, 9, 5, 6]))
    assert round(d2, 4) == 0.2139 and round(p2, 4) == 0.1424, 'статистика D и p-значение'
    assert values.equals(before), 'Series изменился'

    print('Все тесты прошли успешно!')

In [ ]:
lilliefors_test_tests()

In [ ]:
# Проверка: возраст машины водителей области и городского округа Красноармейск через lilliefors_test; печатаются D и p обоих до тысячных, в тест вводятся D по области, p по городскому округу и решение по городскому округу

df_check = pd.read_csv('../datasets/legal/dtp-moscow-region.csv', index_col='id', parse_dates=['datetime'])
people_check = pd.read_csv('../datasets/legal/dtp-moscow-region-participants.csv')
people_check['experience_clean'] = people_check['experience'].replace([95, 96, 99], np.nan)
drivers_check = people_check[people_check['role'] == 'Водитель'].merge(df_check, left_on='dtp_id', right_index=True)
drivers_check['vehicle_age'] = drivers_check['datetime'].dt.year - drivers_check['vehicle_year']
drivers_check['vehicle_age'] = drivers_check['vehicle_age'].where(drivers_check['vehicle_age'] <= 100)

d, p = lilliefors_test(drivers_check['vehicle_age'])
print(round(d, 3), round(p, 3))
town = drivers_check[drivers_check['district'] == 'Красноармейск']
d, p = lilliefors_test(town['vehicle_age'])
print(round(d, 3), round(p, 3))

### Б7. Асимметрия после преобразований

Корень и логарифм поджимают правый хвост: большие значения сжимаются сильнее малых. Логарифм берётся от x + 1, `np.log1p`: у нуля логарифма нет. Отрицательным значениям не подходит ни то ни другое, и функция сообщает об этом ошибкой, а не молча возвращает `nan`; ноль допустим.

Реализуйте функцию `transform_skew(values)`, которая сравнивает асимметрию до и после логарифма и корня. Аргументы: `values` — столбец неотрицательных чисел, Series; пропуски допускаются. Возвращаемое значение: объект Series из трёх чисел с именами «как есть», «log1p», «sqrt»: асимметрия skew() pandas у values, np.log1p(values) и np.sqrt(values); пропуски не учитываются. Если среди значений есть отрицательные — ValueError. Series не меняется.

Подсказка: раздел 6 ноутбука занятия: `np.log1p(values).skew()`, `np.sqrt(values).skew()`; проверка `if (values < 0).any(): raise ValueError(...)` в начале функции. `np.log` даёт минус бесконечность у нуля, и асимметрия становится `nan`.

Проверка в ноутбуке: число раненых в ДТП области через transform_skew; печатаются три асимметрии до сотых, все три вводятся в тест.

In [ ]:
def transform_skew(values):
    """
    Сравнивает асимметрию до и после логарифма и корня.

    Корень и логарифм поджимают правый хвост: большие значения сжимаются сильнее
    малых. Логарифм берётся от x + 1, `np.log1p`: у нуля логарифма нет.
    Отрицательным значениям не подходит ни то ни другое, и функция сообщает об
    этом ошибкой, а не молча возвращает `nan`; ноль допустим.

    Аргументы:
        values: Столбец неотрицательных чисел, Series; пропуски допускаются.

    Возвращаемое значение:
        Объект Series из трёх чисел с именами «как есть», «log1p», «sqrt»:
        асимметрия skew() pandas у values, np.log1p(values) и np.sqrt(values);
        пропуски не учитываются. Если среди значений есть отрицательные —
        ValueError. Series не меняется.
    """

    pass

In [ ]:
def transform_skew_tests():
    values = pd.Series([0, 1, 1, 2, 3, 5, 8, 13, 40, None])
    before = values.copy()
    result = transform_skew(values)
    assert isinstance(result, pd.Series), 'функция должна возвращать Series'
    assert result.index.tolist() == ['как есть', 'log1p', 'sqrt'], 'имена как в условии, в том же порядке'
    assert result.round(3).tolist() == [2.451, 0.616, 1.363], 'асимметрия skew() pandas до и после np.log1p и np.sqrt; ноль допустим'
    try:
        transform_skew(pd.Series([1, -2, 3]))
    except ValueError:
        pass
    else:
        raise AssertionError('отрицательные значения должны давать ValueError')
    assert values.equals(before), 'Series изменился'

    print('Все тесты прошли успешно!')

In [ ]:
transform_skew_tests()

In [ ]:
# Проверка: число раненых в ДТП области через transform_skew; печатаются три асимметрии до сотых, все три вводятся в тест

df_check = pd.read_csv('../datasets/legal/dtp-moscow-region.csv', index_col='id', parse_dates=['datetime'])

print(transform_skew(df_check['injured_count']).round(2))

### Б8. Средние случайных выборок

Опыт центральной предельной теоремы: средние многих выборок одного размера распределены почти нормально с разбросом σ/√n. Генератор `np.random.default_rng(seed)` создаётся один раз, до цикла: тогда при одном seed результат у всех одинаковый, а выборки разные. Выборки с возвращением — `rng.choice(данные, size)`: одно значение может попасть в выборку дважды.

Реализуйте функцию `sample_means(values, size, repeats, seed)`, которая считает средние многих случайных выборок с возвращением. Аргументы: `values` — столбец чисел, Series; пропуски допускаются; `size` — размер одной выборки, целое число; `repeats` — число выборок, целое число; `seed` — зерно генератора, целое число. Возвращаемое значение: объект Series длины repeats с индексом 0, 1, 2, …: среднее каждой из repeats выборок размера size, вытянутых с возвращением из значений values без пропусков генератором np.random.default_rng(seed) — rng.choice(данные, size) подряд repeats раз или одним вызовом rng.choice(данные, size=(repeats, size)); при одном seed результат одинаковый. Series не меняется.

Подсказка: раздел 5 ноутбука занятия: `rng = np.random.default_rng(seed)` до цикла; `data = values.dropna().to_numpy()`; `pd.Series([rng.choice(data, size).mean() for _ in range(repeats)])`. Генератор, созданный внутри цикла, каждый раз тянет одну и ту же выборку; `replace=False` — выборки без возвращения, другие числа.

Проверка в ноутбуке: возраст машины водителей области через sample_means, 500 выборок по 30 с seed=2; печатаются разброс средних и σ/√n до тысячных, оба вводятся в тест.

In [ ]:
def sample_means(values, size, repeats, seed):
    """
    Считает средние многих случайных выборок с возвращением.

    Опыт центральной предельной теоремы: средние многих выборок одного размера
    распределены почти нормально с разбросом σ/√n. Генератор
    `np.random.default_rng(seed)` создаётся один раз, до цикла: тогда при одном
    seed результат у всех одинаковый, а выборки разные. Выборки с возвращением —
    `rng.choice(данные, size)`: одно значение может попасть в выборку дважды.

    Аргументы:
        values: Столбец чисел, Series; пропуски допускаются.
        size: Размер одной выборки, целое число.
        repeats: Число выборок, целое число.
        seed: Зерно генератора, целое число.

    Возвращаемое значение:
        Объект Series длины repeats с индексом 0, 1, 2, …: среднее каждой из
        repeats выборок размера size, вытянутых с возвращением из значений
        values без пропусков генератором np.random.default_rng(seed) —
        rng.choice(данные, size) подряд repeats раз или одним вызовом
        rng.choice(данные, size=(repeats, size)); при одном seed результат
        одинаковый. Series не меняется.
    """

    pass

In [ ]:
def sample_means_tests():
    values = pd.Series([1, 2, 3, 4, 5, None, 10])
    before = values.copy()
    result = sample_means(values, 4, 6, 3)
    assert isinstance(result, pd.Series), 'функция должна возвращать Series'
    assert result.index.tolist() == [0, 1, 2, 3, 4, 5], 'длина repeats, индекс 0, 1, 2, …'
    assert result.notna().all(), 'выборки из значений без пропусков'
    assert result.tolist() == [2.5, 5.25, 1.75, 2.5, 3.0, 5.0], 'средние выборок с возвращением генератором np.random.default_rng(seed), созданным один раз'
    assert sample_means(values, 4, 6, 3).equals(result), 'при одном seed результат одинаковый'
    assert sample_means(values, 4, 6, 4).tolist() == [7.25, 7.75, 3.0, 3.75, 5.25, 4.25], 'зерно генератора — seed'
    assert sample_means(values, 2, 3, 3).tolist() == [3.0, 2.0, 3.5], 'размер выборки — size, число выборок — repeats'
    assert values.equals(before), 'Series изменился'

    print('Все тесты прошли успешно!')

In [ ]:
sample_means_tests()

In [ ]:
# Проверка: возраст машины водителей области через sample_means, 500 выборок по 30 с seed=2; печатаются разброс средних и σ/√n до тысячных, оба вводятся в тест

df_check = pd.read_csv('../datasets/legal/dtp-moscow-region.csv', index_col='id', parse_dates=['datetime'])
people_check = pd.read_csv('../datasets/legal/dtp-moscow-region-participants.csv')
people_check['experience_clean'] = people_check['experience'].replace([95, 96, 99], np.nan)
drivers_check = people_check[people_check['role'] == 'Водитель'].merge(df_check, left_on='dtp_id', right_index=True)
drivers_check['vehicle_age'] = drivers_check['datetime'].dt.year - drivers_check['vehicle_year']
drivers_check['vehicle_age'] = drivers_check['vehicle_age'].where(drivers_check['vehicle_age'] <= 100)

means = sample_means(drivers_check['vehicle_age'], 30, 500, 2)
print(round(means.std(), 3), round(drivers_check['vehicle_age'].std() / np.sqrt(30), 3))

## Часть В. Задачи повышенной сложности

### В1. Отчёт «что дальше»

Таблица решений раздела 6 ноутбука занятия как функция. Сначала размер выборки: при 30 значениях и больше годится t-тест, отвергнута нормальность или нет, — по центральной предельной теореме среднее распределено почти нормально. При меньшем числе значений — решение теста. Если нормальность отвергнута, нужен непараметрический тест: знаковый для одной выборки, Манна–Уитни для двух групп. Если не отвергнута, это мало что значит: малая выборка не видит отклонения, и решают по Q–Q графику и асимметрии. Функция вызывает normality_table из Б5 и добавляет один столбец.

Реализуйте функцию `normality_report(df, columns, alpha=0.05)`, которая добавляет к таблице нормальности столбец «что дальше». Аргументы: `df` — таблица; `columns` — список имён числовых столбцов; `alpha` — уровень значимости, число; по умолчанию 0.05. Возвращаемое значение: объект DataFrame — таблица normality_table(df, columns, alpha) с пятым столбцом «что дальше»: строка «t-тест», если n ≥ 30; при n < 30 — «непараметрический тест», если решение «отвергаем», и «мало данных: решать по Q–Q графику и асимметрии», если «не отвергаем». Таблица df не меняется.

Подсказка: `report = normality_table(df, columns, alpha)`; два вложенных `np.where`: внешний — по маске `report['n'] >= 30`, внутренний — по решению; или список через условное выражение в цикле по `zip(report['n'], report['решение'])`. Ровно 30 значений — уже t-тест; метки — ровно как в условии. Знаковый тест — на семинаре 12, Манна–Уитни — на семинаре 13.

Проверка в ноутбуке: ДТП городского округа Протвино через normality_report по числу участников и числу раненых; печатается отчёт, в тест вводятся p числа участников до тысячных и «что дальше» в обеих строках.

In [ ]:
def normality_report(df, columns, alpha=0.05):
    """
    Добавляет к таблице нормальности столбец «что дальше».

    Таблица решений раздела 6 ноутбука занятия как функция. Сначала размер
    выборки: при 30 значениях и больше годится t-тест, отвергнута нормальность
    или нет, — по центральной предельной теореме среднее распределено почти
    нормально. При меньшем числе значений — решение теста. Если нормальность
    отвергнута, нужен непараметрический тест: знаковый для одной выборки,
    Манна–Уитни для двух групп. Если не отвергнута, это мало что значит: малая
    выборка не видит отклонения, и решают по Q–Q графику и асимметрии. Функция
    вызывает normality_table из Б5 и добавляет один столбец.

    Аргументы:
        df: Таблица.
        columns: Список имён числовых столбцов.
        alpha: Уровень значимости, число; по умолчанию 0.05.

    Возвращаемое значение:
        Объект DataFrame — таблица normality_table(df, columns, alpha) с пятым
        столбцом «что дальше»: строка «t-тест», если n ≥ 30; при n < 30 —
        «непараметрический тест», если решение «отвергаем», и «мало данных:
        решать по Q–Q графику и асимметрии», если «не отвергаем». Таблица df не
        меняется.
    """

    pass

In [ ]:
def normality_report_tests():
    big = [1] * 18 + [2, 2, 2, 3, 3, 4, 5, 6, 8, 12, 20, 35]
    small_skew = [1] * 12 + [2, 2, 3, 3, 4, 5, 8, 13, 21, 34, 55, 89, 144] + [None] * 5
    small_norm = [31, 37, 40, 42, 44, 45, 47, 48, 49, 50, 51, 52, 53, 55, 56, 58, 60, 63, 69, 46, 54, 50] + [None] * 8
    df = pd.DataFrame({'big': big, 'small_skew': small_skew, 'small_norm': small_norm})
    before = df.copy()
    report = normality_report(df, ['big', 'small_skew', 'small_norm'])
    assert isinstance(report, pd.DataFrame), 'функция должна возвращать DataFrame'
    assert report.columns.tolist() == ['n', 'асимметрия', 'p', 'решение', 'что дальше'], 'таблица normality_table и столбец «что дальше»'
    assert report['n'].tolist() == [30, 25, 22], 'n — число значений без пропусков'
    assert report['решение'].tolist() == ['отвергаем', 'отвергаем', 'не отвергаем']
    assert report.loc['big', 'что дальше'] == 't-тест', 'n ≥ 30 — t-тест, хотя нормальность отвергнута; 30 — уже t-тест'
    assert report.loc['small_skew', 'что дальше'] == 'непараметрический тест', 'n < 30 и «отвергаем» — непараметрический тест'
    assert report.loc['small_norm', 'что дальше'] == 'мало данных: решать по Q–Q графику и асимметрии', 'n < 30 и «не отвергаем» — мало данных'
    assert normality_report(df, ['small_norm'], alpha=0.99)['что дальше'].tolist() == ['непараметрический тест'], 'уровень значимости alpha передаётся в normality_table'
    assert df.equals(before), 'таблица изменилась'

    print('Все тесты прошли успешно!')

In [ ]:
normality_report_tests()

In [ ]:
# Проверка: ДТП городского округа Протвино через normality_report по числу участников и числу раненых; печатается отчёт, в тест вводятся p числа участников до тысячных и «что дальше» в обеих строках

df_check = pd.read_csv('../datasets/legal/dtp-moscow-region.csv', index_col='id', parse_dates=['datetime'])

town = df_check[df_check['district'] == 'Протвино']
print(normality_report(town, ['participants_count', 'injured_count']))

### В2. Панели Q–Q

Q–Q графики к таблице раздела «Нормальность»: по панели на столбец, рядом, чтобы сравнивать формы. Каждая панель — функция qq_plot из Б3 на своих осях, заголовок панели — имя столбца. Один столбец — тоже ряд из одной панели: `plt.subplots(1, 1)` возвращает не массив, а одни оси, и цикл по ним ломается; `squeeze=False` всегда даёт двумерный массив осей.

Реализуйте функцию `qq_panels(df, columns)`, которая рисует Q–Q графики нескольких столбцов на панелях в один ряд. Аргументы: `df` — таблица; `columns` — список имён числовых столбцов. Возвращаемое значение: объект Figure из plt.subplots(1, len(columns)) с панелью на каждый столбец columns в том же порядке слева направо: на панели qq_plot(df[столбец], столбец, ax=…) — точки и прямая, заголовок — имя столбца; fig.tight_layout(). Таблица не меняется.

Подсказка: `fig, axes = plt.subplots(1, len(columns), figsize=(4 * len(columns), 4), squeeze=False)`; цикл `for ax, column in zip(axes[0], columns):`; `return fig`.

Проверка в ноутбуке: стаж и возраст машины водителей области через qq_panels; печатаются заголовки панелей и наклоны прямых до сотых, оба наклона вводятся в тест.

In [ ]:
def qq_panels(df, columns):
    """
    Рисует Q–Q графики нескольких столбцов на панелях в один ряд.

    Q–Q графики к таблице раздела «Нормальность»: по панели на столбец, рядом,
    чтобы сравнивать формы. Каждая панель — функция qq_plot из Б3 на своих осях,
    заголовок панели — имя столбца. Один столбец — тоже ряд из одной панели:
    `plt.subplots(1, 1)` возвращает не массив, а одни оси, и цикл по ним
    ломается; `squeeze=False` всегда даёт двумерный массив осей.

    Аргументы:
        df: Таблица.
        columns: Список имён числовых столбцов.

    Возвращаемое значение:
        Объект Figure из plt.subplots(1, len(columns)) с панелью на каждый
        столбец columns в том же порядке слева направо: на панели
        qq_plot(df[столбец], столбец, ax=…) — точки и прямая, заголовок — имя
        столбца; fig.tight_layout(). Таблица не меняется.
    """

    pass

In [ ]:
def qq_panels_tests():
    df = pd.DataFrame({'a': [3, 1, 4, 1, 5, None, 9, 2], 'b': [10, 20, None, 15, 12, 18, 30, 11], 'name': list('qwertyui')})
    before = df.copy()
    fig = qq_panels(df, ['b', 'a'])
    assert isinstance(fig, plt.Figure), 'функция должна возвращать Figure'
    assert len(fig.axes) == 2, 'панель на каждый столбец'
    left, right = sorted(fig.axes, key=lambda ax: ax.get_position().x0)
    assert abs(left.get_position().y0 - right.get_position().y0) < 0.01, 'панели в один ряд'
    assert [left.get_title(), right.get_title()] == ['b', 'a'], 'заголовок панели — имя столбца, порядок как в columns'
    assert all(len(ax.lines) == 2 for ax in fig.axes), 'на каждой панели точки и прямая'
    assert left.lines[0].get_ydata().tolist() == [10, 11, 12, 15, 18, 20, 30], 'на панели Q–Q график своего столбца без пропусков'
    assert left.get_xlabel() == 'Квантили нормального распределения', 'панели рисует qq_plot'
    assert len(qq_panels(df, ['a']).axes) == 1, 'один столбец — одна панель'
    assert df.equals(before), 'таблица изменилась'
    plt.close('all')

    print('Все тесты прошли успешно!')

In [ ]:
qq_panels_tests()

In [ ]:
# Проверка: стаж и возраст машины водителей области через qq_panels; печатаются заголовки панелей и наклоны прямых до сотых, оба наклона вводятся в тест

df_check = pd.read_csv('../datasets/legal/dtp-moscow-region.csv', index_col='id', parse_dates=['datetime'])
people_check = pd.read_csv('../datasets/legal/dtp-moscow-region-participants.csv')
people_check['experience_clean'] = people_check['experience'].replace([95, 96, 99], np.nan)
drivers_check = people_check[people_check['role'] == 'Водитель'].merge(df_check, left_on='dtp_id', right_index=True)
drivers_check['vehicle_age'] = drivers_check['datetime'].dt.year - drivers_check['vehicle_year']
drivers_check['vehicle_age'] = drivers_check['vehicle_age'].where(drivers_check['vehicle_age'] <= 100)

fig = qq_panels(drivers_check, ['experience_clean', 'vehicle_age'])
for ax in fig.axes:
    x, y = ax.lines[1].get_xdata(), ax.lines[1].get_ydata()
    print(ax.get_title(), round((y[-1] - y[0]) / (x[-1] - x[0]), 2))

### В3. Центральная предельная теорема на рисунке

Чем больше выборка, тем ближе распределение её среднего к нормальному: на выборках по 5 значений хвост скошенного столбца ещё виден, на 100 — колокол. Асимметрия в заголовке панели говорит то же числом. Средние считает функция sample_means из Б8 с одним и тем же seed для каждой панели.

Реализуйте функцию `clt_demo(values, sizes=(5, 30, 100), repeats=1000, seed=1)`, которая рисует распределения средних выборок разного размера. Аргументы: `values` — столбец чисел, Series; пропуски допускаются; `sizes` — размеры выборок, кортеж целых чисел; по умолчанию (5, 30, 100); `repeats` — число выборок на панель, целое число; по умолчанию 1000; `seed` — зерно генератора, целое число; по умолчанию 1. Возвращаемое значение: объект Figure из plt.subplots(1, len(sizes)) с панелью на каждый размер из sizes слева направо: гистограмма sns.histplot средних sample_means(values, size, repeats, seed), по оси Y — число выборок; заголовок панели f'n = {size}, асимметрия {s:.2f}', где s — skew() этих средних; подписи осей «Среднее выборки» и «Число выборок»; fig.tight_layout(). Series не меняется.

Подсказка: `fig, axes = plt.subplots(1, len(sizes), figsize=(4 * len(sizes), 4), squeeze=False)`; в цикле по `zip(axes[0], sizes)` — `means = sample_means(values, size, repeats, seed)`, `sns.histplot(x=means, ax=ax)`, заголовок через f-строку с `:.2f`.

Проверка в ноутбуке: возраст машины водителей области через clt_demo с параметрами по умолчанию; печатаются заголовки панелей, в тест вводятся асимметрии для n = 5 и n = 100.

In [ ]:
def clt_demo(values, sizes=(5, 30, 100), repeats=1000, seed=1):
    """
    Рисует распределения средних выборок разного размера.

    Чем больше выборка, тем ближе распределение её среднего к нормальному: на
    выборках по 5 значений хвост скошенного столбца ещё виден, на 100 — колокол.
    Асимметрия в заголовке панели говорит то же числом. Средние считает функция
    sample_means из Б8 с одним и тем же seed для каждой панели.

    Аргументы:
        values: Столбец чисел, Series; пропуски допускаются.
        sizes: Размеры выборок, кортеж целых чисел; по умолчанию (5, 30, 100).
        repeats: Число выборок на панель, целое число; по умолчанию 1000.
        seed: Зерно генератора, целое число; по умолчанию 1.

    Возвращаемое значение:
        Объект Figure из plt.subplots(1, len(sizes)) с панелью на каждый размер
        из sizes слева направо: гистограмма sns.histplot средних
        sample_means(values, size, repeats, seed), по оси Y — число выборок;
        заголовок панели f'n = {size}, асимметрия {s:.2f}', где s — skew() этих
        средних; подписи осей «Среднее выборки» и «Число выборок»;
        fig.tight_layout(). Series не меняется.
    """

    pass

In [ ]:
def clt_demo_tests():
    values = pd.Series([1, 2, 2, 3, 4, 7, 15, None])
    before = values.copy()
    fig = clt_demo(values, sizes=(2, 6), repeats=40, seed=5)
    assert isinstance(fig, plt.Figure), 'функция должна возвращать Figure'
    assert len(fig.axes) == 2, 'панель на каждый размер выборки'
    left, right = sorted(fig.axes, key=lambda ax: ax.get_position().x0)
    assert abs(left.get_position().y0 - right.get_position().y0) < 0.01, 'панели в один ряд'
    assert [left.get_title(), right.get_title()] == ['n = 2, асимметрия 1.11', 'n = 6, асимметрия 0.38'], 'заголовок с асимметрией средних sample_means(values, size, repeats, seed), один seed для всех панелей'
    assert all(sum(p.get_height() for p in ax.patches) == 40 for ax in fig.axes), 'гистограмма числа выборок: высоты в сумме repeats'
    assert left.get_xlabel() == 'Среднее выборки' and left.get_ylabel() == 'Число выборок', 'подписи осей'
    titles = [ax.get_title() for ax in sorted(clt_demo(values).axes, key=lambda ax: ax.get_position().x0)]
    assert titles == ['n = 5, асимметрия 0.65', 'n = 30, асимметрия 0.34', 'n = 100, асимметрия 0.16'], 'размеры, число выборок и seed по умолчанию'
    assert values.equals(before), 'Series изменился'
    plt.close('all')

    print('Все тесты прошли успешно!')

In [ ]:
clt_demo_tests()

In [ ]:
# Проверка: возраст машины водителей области через clt_demo с параметрами по умолчанию; печатаются заголовки панелей, в тест вводятся асимметрии для n = 5 и n = 100

df_check = pd.read_csv('../datasets/legal/dtp-moscow-region.csv', index_col='id', parse_dates=['datetime'])
people_check = pd.read_csv('../datasets/legal/dtp-moscow-region-participants.csv')
people_check['experience_clean'] = people_check['experience'].replace([95, 96, 99], np.nan)
drivers_check = people_check[people_check['role'] == 'Водитель'].merge(df_check, left_on='dtp_id', right_index=True)
drivers_check['vehicle_age'] = drivers_check['datetime'].dt.year - drivers_check['vehicle_year']
drivers_check['vehicle_age'] = drivers_check['vehicle_age'].where(drivers_check['vehicle_age'] <= 100)

fig = clt_demo(drivers_check['vehicle_age'])
for ax in fig.axes:
    print(ax.get_title())

## Необязательная задача. Раздел «Нормальность» этапа 3 для своего датасета

Этап 3 проекта выдаётся на семинаре 12 и начинается с нормальности: прежде чем проверять гипотезы о средних, нужно решить, какой тест для них годится. Раздел для своего датасета сейчас — готовая часть отчёта этапа 3: он переходит в отчёт как есть и сдаётся вместе с ним на семинаре 14.

Состав — функциями частей Б и В: таблица `normality_report` по числовым столбцам; Q–Q графики `qq_panels` по тем же столбцам; при желании сводка `normal_summary` и гистограмма `hist_with_normal` по главному столбцу. Вывод — по каждому столбцу словами, как в разделе 7 ноутбука занятия: решение теста, размер выборки, что дальше — t-тест, непараметрический тест или решение по Q–Q графику и асимметрии — и почему; для столбцов, где среднее не имеет смысла, как у часа, — оговорка. p меньше 0,001 записываются как «p < 0,001».

Если датасета ещё нет или в нём нет числовых столбцов, возьмите таблицы Брянской области `datasets/legal/dtp-bryansk.csv` и `datasets/legal/dtp-bryansk-participants.csv`: в упражнениях они не использовались; три столбца, разобранные в ноутбуке занятия, — стаж, возраст машины и час — замените другими.

В ответ в тест Moodle вставьте код, таблицу и вывод; вложений у этого вопроса нет. Балл ставит преподаватель; автоматической проверки нет.

In [ ]:
path = '../datasets/legal/dtp-bryansk.csv'   # путь к своему файлу; пока таблица ДТП Брянской области
data = pd.read_csv(path)

print(data.shape)
print(data.dtypes)

In [ ]:
# таблица нормальности по числовым столбцам

In [ ]:
# Q–Q графики тех же столбцов

In [ ]:
# сводка и гистограмма главного столбца

Вывод раздела «Нормальность»

| Столбец | n | Асимметрия | Тест и p | Решение | Что дальше и почему |
|---|---|---|---|---|---|
| | | | | | |